# SmolLM2-135M Memory Fusion — Sequential Acceptance Training v2

This is the **fixed/resumable** version. It replaces one Transformer attention layer at a time and never silently advances past a bad replacement.

Important fixes:
- a layer that misses the acceptance gate keeps its trained weights and resumes from them on the next run;
- expected scientific stops no longer become opaque `CalledProcessError`s;
- subprocess output is unbuffered and streamed live;
- the real child traceback is saved to Google Drive as `sequential_last_error.json` if there is a software error;
- Google Drive stores accepted layers **and the current not-yet-accepted layer**.


In [ ]:
import os, sys, pathlib, subprocess, json, time
import torch

subprocess.run(['nvidia-smi'], check=False)
REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if not REPO_DIR.exists():
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)], check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'checkout','main'], check=False)
    subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'], check=True)

subprocess.run([
    sys.executable,'-m','pip','install','-q',
    '-e',str(REPO_DIR),
    'transformers==4.57.6','datasets>=3,<5','huggingface_hub>=0.34,<2',
    'pandas','matplotlib'
], check=True)
print('Python:', sys.executable)
print('Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/TinyCeNN-LM')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Persistent root:', DRIVE_ROOT)


## Settings
Keep these defaults for the first run. If a layer needs more training, simply rerun the training cell (or reconnect later and run the notebook again). It will continue the same layer from its saved weights.


In [ ]:
BASE_MODEL = 'HuggingFaceTB/SmolLM2-135M'
MEMORY_RANK = 64
FEATURE_DIM = 32
CONTEXT_LENGTH = 128
SEED = 73
RESUME = True

MIN_LAYER_STEPS = 50
MAX_LAYER_STEPS = 300
CHECK_EVERY = 25
LAYER_LR = 2e-4
TEACHER_ALPHA_START = 0.90
TEACHER_ALPHA_END = 0.00

ACCEPT_NMSE = 0.20
ACCEPT_COSINE = 0.90
ACCEPT_INCREMENTAL_DELTA_NLL = 0.015
ACCEPT_CUMULATIVE_DELTA_NLL = 0.05

CORE_O_TOKENS = 50_000
NORM_TOKENS = 50_000
FULL_TOKENS = 100_000
MAX_RUNTIME_MINUTES = 240

# The trainer attempts up to four 300-step rounds on the current layer in one
# session. If it still misses the gate it exits normally and resumes later.
MAX_ROUNDS_PER_RUN = 4

OUTPUT_DIR = DRIVE_ROOT / f'smollm2-memory-fusion-sequential-r{MEMORY_RANK}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output:', OUTPUT_DIR)


## Train / resume
The output below is streamed live. A return code of 0 can mean either **complete**, **a safely paused run**, or **the current layer needs more training**. Check the status cell afterward.


In [ ]:
cmd = [
    sys.executable, '-u',
    str(REPO_DIR / 'scripts' / 'train_smollm2_memory_fusion_sequential_v2.py'),
    '--base-model', BASE_MODEL,
    '--output-dir', str(OUTPUT_DIR),
    '--memory-rank', str(MEMORY_RANK),
    '--feature-dim', str(FEATURE_DIM),
    '--context-length', str(CONTEXT_LENGTH),
    '--seed', str(SEED),
    '--min-layer-steps', str(MIN_LAYER_STEPS),
    '--max-layer-steps', str(MAX_LAYER_STEPS),
    '--check-every', str(CHECK_EVERY),
    '--layer-lr', str(LAYER_LR),
    '--teacher-alpha-start', str(TEACHER_ALPHA_START),
    '--teacher-alpha-end', str(TEACHER_ALPHA_END),
    '--accept-nmse', str(ACCEPT_NMSE),
    '--accept-cosine', str(ACCEPT_COSINE),
    '--accept-incremental-delta-nll', str(ACCEPT_INCREMENTAL_DELTA_NLL),
    '--accept-cumulative-delta-nll', str(ACCEPT_CUMULATIVE_DELTA_NLL),
    '--core-o-tokens', str(CORE_O_TOKENS),
    '--norm-tokens', str(NORM_TOKENS),
    '--full-tokens', str(FULL_TOKENS),
    '--max-runtime-minutes', str(MAX_RUNTIME_MINUTES),
]
cmd.append('--resume' if RESUME else '--no-resume')
cmd.append('--strict-acceptance')

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['SEQUENTIAL_MAX_ROUNDS_PER_RUN'] = str(MAX_ROUNDS_PER_RUN)
log_path = OUTPUT_DIR / 'last_colab_run.log'
print(' '.join(cmd))
print('Log:', log_path)

with log_path.open('w', encoding='utf-8') as log:
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env
    )
    tail = []
    for line in proc.stdout:
        print(line, end='')
        log.write(line); log.flush()
        tail.append(line.rstrip())
        if len(tail) > 120: tail.pop(0)
    rc = proc.wait()

print('\nProcess return code:', rc)
if rc != 0:
    print('\n--- LAST CHILD OUTPUT ---')
    print('\n'.join(tail))
    error_file = OUTPUT_DIR / 'sequential_last_error.json'
    if error_file.exists():
        print('\n--- SAVED SOFTWARE ERROR ---')
        print(error_file.read_text())
    raise RuntimeError(f'Sequential trainer failed with return code {rc}. See the full child traceback above and {log_path}.')


## Current status


In [ ]:
status_path = OUTPUT_DIR / 'sequential_run_status.json'
progress_path = OUTPUT_DIR / 'sequential_progress.json'
in_progress_path = OUTPUT_DIR / 'sequential_in_progress.json'
report_path = OUTPUT_DIR / 'sequential_training_report.json'

if status_path.exists():
    status = json.loads(status_path.read_text())
    print(json.dumps(status, indent=2))
else:
    status = {'status':'unknown'}
    print('No status file yet.')

if progress_path.exists():
    p = json.loads(progress_path.read_text())
    print('Accepted layers:', p.get('accepted_layers', []))
if in_progress_path.exists():
    p = json.loads(in_progress_path.read_text())
    print('Current layer:', p.get('current_layer'))
    print('Rounds completed:', p.get('rounds_completed'))
    reports = p.get('layer_reports', [])
    if reports:
        print('Last acceptance report:', json.dumps(reports[-1], indent=2))

print('\nInterpretation:')
if status.get('status') == 'current_layer_needs_more_training':
    print('✅ No crash. The current layer did not pass yet. Rerun the training cell; it will continue from these exact weights.')
elif status.get('status') == 'paused_runtime_budget':
    print('✅ Safely paused. Reconnect/rerun with RESUME=True.')
elif status.get('status') == 'complete':
    print('🏁 Full 30-layer conversion and integrated training completed.')
elif status.get('status') == 'software_error':
    print('❌ Real software error. Inspect sequential_last_error.json and last_colab_run.log.')


## Layer progress table


In [ ]:
import pandas as pd
reports = []
if report_path.exists():
    reports = json.loads(report_path.read_text()).get('layer_reports', [])
elif in_progress_path.exists():
    reports = json.loads(in_progress_path.read_text()).get('layer_reports', [])
elif progress_path.exists():
    reports = json.loads(progress_path.read_text()).get('layer_reports', [])
df = pd.DataFrame(reports)
display(df.tail(40) if not df.empty else df)
